# Phase 3: checking one prepared batch

**CRISP-DM stage:** Data preparation

A batch is one tray of photos processed together. Before any model runs, this checks one tray exactly as a model would receive it: are the parts visible, and do the masks sit on the defects? A mistake here would quietly spoil every later result.

**Runs on:** Kaggle with the GPU off, because no model runs, and internet on, to install anomalib. On the laptop it only checks the code, on the few sample photos there.

**Which photos:** half A only, in `regular` lighting. Half B stays sealed.

**Figures with dataset images:** files named `gallery_*` show real parts, so they stay on the laptop and out of git. Each figure is saved and closed, not displayed.

In [ ]:
!pip install anomalib==2.7.0

## 1. Find the photos

The part numbers come from `half_a_files.csv`. They are written here because Kaggle cannot read files from the laptop. The first 6 defective and first 2 good half A parts are used, so the choice is repeatable. 8 is the same batch size as the GPU check, and the mix matters because a good photo's mask must be empty.

`MVTecAD2` is anomalib's loader. `root` is the outer product folder, `category` is the inner one, and `num_workers=0` reads photos in the main process, which is simplest for a small check. `setup` builds the lists of photos. `prepare_data` is not called here, because it could try to download.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from anomalib.data import MVTecAD2
from anomalib.models import Padim, WinClip
from torch.utils.data import DataLoader, Subset

if Path("/kaggle/input").exists():
    INPUT_FOLDER = Path("/kaggle/input")
else:
    INPUT_FOLDER = Path("data/sample")

HALF_A_PARTS = {
    "sheet_metal": {"bad": ["000", "004", "005", "007", "009", "012"], "good": ["001", "002"]},
    "vial": {"bad": ["001", "008", "010", "011", "012", "013"], "good": ["000", "002"]},
}
FIGURE_DPI = 150
PHOTOS_ACROSS = 4
TOLERANCE = 0.0001

product_folders = {}
for train_folder in INPUT_FOLDER.rglob("train"):
    if train_folder.parent.name in HALF_A_PARTS:
        product_folders[train_folder.parent.name] = train_folder.parent

datasets = {}
trays = {}
for product_name in HALF_A_PARTS:
    product_folder = product_folders[product_name]
    datamodule = MVTecAD2(root=product_folder.parent, category=product_name, num_workers=0)
    datamodule.setup()
    dataset = datamodule.test_public_data
    datasets[product_name] = dataset

    chosen_positions = []
    image_paths = dataset.samples["image_path"].tolist()
    for position in range(len(image_paths)):
        path = Path(image_paths[position])
        label = path.parent.name
        if path.name.endswith("_regular.png") and path.stem.split("_")[0] in HALF_A_PARTS[product_name][label]:
            chosen_positions.append(position)
    trays[product_name] = chosen_positions
    print(product_name, "photos in the tray:", len(chosen_positions))

## 2. Prepare the batch and check it

`DataLoader` builds the tray, and `collate_fn` is the dataset's own function for joining photos into one batch. `Subset` limits it to the chosen photos.

Each recipe is squash then normalize. `recipe(batch.image, batch.gt_mask)` is the same call anomalib makes before a model sees a batch.

- **256 with ImageNet numbers:** PaDiM, PatchCore and Reverse Distillation in the main experiments.
- **240 with CLIP numbers:** WinCLIP, whose size is fixed.
- **512 with ImageNet numbers:** the resolution test, numbers only.

**What correct values look like:**
- The shape is `[8, 3, size, size]`: 8 photos, 3 channels, height and width.
- Normalized values run from about -2.12 (black) to 2.64 (white) with ImageNet numbers, and from about -1.79 to 2.15 with CLIP numbers. CLIP's bicubic resize can overshoot these a little near sharp edges.
- Masks hold false and true only. A good photo's mask is empty, and a defective photo's is not.

**Undoing normalization** reverses the sum: `old value = new value x std + mean`. It must return the resized photo, or the recipe is not what we think.

In [ ]:
RECIPES = [
    ("imagenet_256", 256, Padim.configure_pre_processor(image_size=(256, 256)).transform),
    ("clip_240", 240, WinClip.configure_pre_processor().transform),
    ("imagenet_512", 512, Padim.configure_pre_processor(image_size=(512, 512)).transform),
]
RECIPES_DRAWN = ["imagenet_256", "clip_240"]


def undo_normalization(images, normalize_step):
    """Reverse the normalization sum, one channel at a time."""
    means = torch.tensor(normalize_step.mean).view(1, 3, 1, 1)
    stds = torch.tensor(normalize_step.std).view(1, 3, 1, 1)
    return images * stds + means


check_lines = []

for product_name in HALF_A_PARTS:
    dataset = datasets[product_name]
    tray = Subset(dataset, trays[product_name])
    loader = DataLoader(tray, batch_size=len(tray), collate_fn=dataset.collate_fn, shuffle=False)
    batch = next(iter(loader))

    for recipe_name, size, recipe in RECIPES:
        prepared_images, prepared_masks = recipe(batch.image, batch.gt_mask)
        resized_only = recipe.transforms[0](batch.image)
        undone = undo_normalization(prepared_images, recipe.transforms[1])

        expected_shape = [len(tray), 3, size, size]
        if list(prepared_images.shape) != expected_shape:
            raise ValueError("Wrong batch shape: " + str(list(prepared_images.shape)))
        if float((undone - resized_only).abs().max()) > TOLERANCE:
            raise ValueError("Undoing normalization did not return the photo")
        if float(prepared_images.min()) >= 0:
            raise ValueError("No negative values, so normalization may not have happened")
        if prepared_masks.dtype != torch.bool:
            raise ValueError("Mask is not true or false: " + str(prepared_masks.dtype))

        check_lines.append("")
        check_lines.append("=== " + product_name + ", " + recipe_name)
        check_lines.append("batch shape: " + str(list(prepared_images.shape)))
        check_lines.append("mask shape: " + str(list(prepared_masks.shape)))
        check_lines.append("value range after normalization: " + str(round(float(prepared_images.min()), 3))
                           + " to " + str(round(float(prepared_images.max()), 3)))
        check_lines.append("unique mask values: " + str(torch.unique(prepared_masks).tolist()))

        for position in range(len(tray)):
            path = Path(batch.image_path[position])
            label = path.parent.name
            defect_pixels = int(prepared_masks[position].sum())
            if label == "good" and defect_pixels != 0:
                raise ValueError("A good photo has a mask: " + path.name)
            if label == "bad" and defect_pixels == 0:
                raise ValueError("A defective photo has an empty mask: " + path.name)
            check_lines.append("  " + label + " " + path.name + ": defect pixels " + str(defect_pixels))

        if recipe_name in RECIPES_DRAWN:
            # Enough rows to hold every photo, rounding up
            rows = (len(tray) + PHOTOS_ACROSS - 1) // PHOTOS_ACROSS
            figure, axes = plt.subplots(rows, PHOTOS_ACROSS, figsize=(4 * PHOTOS_ACROSS, 4.3 * rows), squeeze=False)
            for panel in range(rows * PHOTOS_ACROSS):
                axis = axes[panel // PHOTOS_ACROSS][panel % PHOTOS_ACROSS]
                axis.axis("off")
                if panel >= len(tray):
                    continue
                path = Path(batch.image_path[panel])
                # All channels hold the same picture once normalization is undone
                axis.imshow(undone[panel, 0].clamp(0, 1), cmap="gray", vmin=0, vmax=1)
                mask_for_outline = prepared_masks[panel].float()
                if int(mask_for_outline.sum()) > 0:
                    axis.contour(mask_for_outline, levels=[0.5], colors="red", linewidths=0.8)
                axis.set_title(path.parent.name + " " + path.name + ", " + str(int(mask_for_outline.sum())) + " defect pixels", fontsize=9)
            figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A, " + recipe_name
                            + ", normalization undone. Red outline: mask")
            figure.savefig("gallery_prepared_batch_" + product_name + "_" + recipe_name + ".png",
                           dpi=FIGURE_DPI, bbox_inches="tight")
            # Closing without showing keeps dataset images out of the notebook's log
            plt.close(figure)

# Numbers only, with no image content, so this file can be committed
with open("prepared_batch_check.txt", "w") as summary_file:
    for line in check_lines:
        summary_file.write(line + "\n")

for line in check_lines:
    print(line)
print("All automatic checks passed")